In [ ]:
## Python Package Import
import sys
import os 
import numpy as np
import pandas as pd
from datetime import datetime

In [ ]:
# copying ancestry predictions and list of related individuals to aux folder on persistent disk
# 2 TB solid state disk at $340 per month but this pipeline has lots of IO, which will save time.

!gsutil -u $GOOGLE_PROJECT cp 'gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv' ./aux
!gsutil -u $GOOGLE_PROJECT cp 'gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/aux/relatedness/relatedness_flagged_samples.tsv' ./aux

In [ ]:
# This and next cell randomly selects AoU IDs to match required population composition

biobank = 'meta_analysis_mixed'
n = 5500 # sample size for LD reference calculations. 

# order is as follows: ["eur", "afr", "amr", "eas", "sas", "mid"]


# Mixed ancestry meta_analysis fractions 
# fractions=[0.705,0.17,0.044,0.081,0,0]

# EUR fraction
fractions=[0,0,0,1,0,0]

counts = [int(round(n*x, 0)) for x in fractions]

pop_fractions = pd.DataFrame({'POP':["eur", "afr", "amr", "eas", "sas", "mid"],
                            'Counts': counts})

# generate an ID list for custom LD score calculation


# reading AoU major continental population predictions
pop = pd.read_csv("./aux/ancestry_preds.tsv", sep = '\t')

# removing related individuals
rel = pd.read_csv("./aux/relatedness_flagged_samples.tsv", sep = '\t')
pop = pop[~pop['research_id'].isin(rel['sample_id'])]

mapper = pop_fractions.set_index('POP')['Counts'].to_dict()

IDs = pop.groupby('ancestry_pred').apply(lambda x: x.sample(n=mapper.get(x.name))).reset_index(drop = True)

In [ ]:
IDs['ancestry_pred'].value_counts()

In [ ]:
%env BIOBANK={biobank}

In [ ]:
env: BIOBANK=meta_analysis_eas
# making a list for plink and saving it to cloud
ID_list = pd.DataFrame({'FID':IDs['research_id'],
                        'IID':IDs['research_id'],
                        'zero':0})

ID_list.to_csv("gs://~/GBMI_meta_analysis_custom_LD_reference/meta_analysis_EAS.idlist", 
               sep = '\t',
               index=False, header=False)

! gsutil -u $GOOGLE_PROJECT cp gs://~/GBMI_meta_analysis_custom_LD_reference/meta_analysis_EAS.idlist ./aux/

In [ ]:
for chrom in range(1,23):
    !plink2 --pfile './reference/chr{chrom}' \
        --clump './gwas/benign_mixed/benign_mixed{chrom}.txt' \
        --clump-p1 5e-8 \
        --clump-p2 1e-5 \
        --clump-r2 0.01 \
        --clump-kb 5000 \
        --clump-allow-overlap \
        --clump-field "p" \
        --clump-snp-field "SNP" \
        --out './out/bng/benign_mixed{chrom}'

In [ ]:
# grouping clumps into groups that are at least 5 MB distant from each other

def group_by_distance(numbers, max_distance):
    if not numbers:
        return []

    sorted_numbers = sorted(numbers)
    groups = []
    current_group = [sorted_numbers[0]]

    for i in range(1, len(sorted_numbers)):
        if sorted_numbers[i] - current_group[-1] < max_distance:
            current_group.append(sorted_numbers[i])
        else:
            groups.append(current_group)
            current_group = [sorted_numbers[i]]
    
    groups.append(current_group) # Add the last group

    return groups

In [ ]:
def indepedent_with_first(refhit, subs):
    
    refhit_list = refhit['SP2'].to_list()[0].split(',')
    
    # clumps indepedent from reference clump
    th = subs[0:0]
    
    # checking reference clump of the group for variant overlap with other candidate clumps in the group 
    for candidate_pos in subs['POS']:
        candidate =  subs[subs['POS'] == candidate_pos]
        candidate_list =  candidate['SP2'].to_list()[0].split(',')
#        candidate_list.extend(candidate['ID'].tolist())

        # Convert to sets and find intersection between two lists of variants in reference and candidate clumps
        common_elements_set = set(refhit_list) & set(candidate_list)
        common_elements_list = list(common_elements_set)

        if(len(common_elements_list) == 0): # if no intesection between variant lists - adding candidate to tophits
            th = pd.concat([th, candidate], ignore_index=True)
    
    return th

In [ ]:

chrom = 1
chrom = str(chrom)
clumps = pd.read_csv('/home/jupyter/workspaces/~/out/bng/benign_mixed' + chrom + '.clumps', sep = '\t')
clumps = clumps.sort_values(by = 'POS')

tophits = clumps[0:0]


for chrom in range(1,23):
    
    chrom = str(chrom)
    file_path = '/home/jupyter/workspaces/~/out/bng/benign_mixed' + chrom + '.clumps'
    if os.path.exists(file_path):
        clumps = pd.read_csv(file_path, sep = '\t')
        clumps = clumps.sort_values(by = 'POS')

        # extracting groups of nearby clumps
        my_list = clumps['POS'].to_list()
        distance_threshold = 5000000
        result_groups = group_by_distance(my_list, distance_threshold)

        for vars in result_groups:

            subs = clumps[clumps['POS'].isin(vars)] # subsetting to variants in the group of nearby clumps

            if subs.shape[0] == 1: # singelton clump adding to tophits right away
                tophits = pd.concat([tophits, subs.iloc[[0]]], ignore_index=True)
            else:
                while subs.shape[0] > 1: # repeat until at least two clumps exist to compare

                    subs = subs.sort_values(by = 'P')

                    # adding the most significant remaining clump to tophits
                    tophits = pd.concat([tophits, subs.iloc[[0]]], ignore_index=True)
                    refhit = subs.iloc[[0]] # setting most significant hit as reference

                    # removing the reference clump from comparison
                    subs = subs.iloc[1:]
                    subs = indepedent_with_first(refhit, subs) # this removes the first (reference) clump and all non-independent with it

                    if subs.shape[0] == 1: # if just one clump is left, it is independent
                        tophits = pd.concat([tophits, subs.iloc[[0]]], ignore_index=True)
                    
tophits = tophits.sort_values(by = ['#CHROM', 'POS'])

tophits.to_csv("./out/final_lists/benign_mixed", 
               sep = '\t',
               index=False, header=False)